# Tutorial 2: **Notebook** - *FVF*

> 🚧 **Under Construction**

**By 
gLayout Team**

__Content creators:__ Subham Pal, Saptarshi Ghosh

__Content reviewers:__ Mehedi Saligne

___
# Tutorial Objectives

This notebook is a tutorial on-

- **LVS (Layout Versus Schematic):**  
  You will learn how to compare your physical layout with the original schematic to ensure they are functionally identical. This process helps catch connectivity or device mismatches before fabrication.

- **Extraction and Simulation:**  
  The tutorial will guide you through extracting parasitic elements from your layout, such as capacitance and resistance, to create a more accurate circuit model. You will then simulate the extracted netlist to analyze and verify the real-world performance of your design.

## **Target** **Block** : **Flipped Voltage Follwer Cell**

A **voltage follower**—also known as a unity-gain buffer or buffer amplifier—is an electronic circuit in which the output voltage precisely follows the input voltage, providing a voltage gain of one. Typically implemented using an operational amplifier (op-amp) with negative feedback, the voltage follower features extremely high input impedance and very low output impedance. This configuration allows it to isolate circuit stages, preventing the loading of the input source and enabling the circuit to drive low-impedance loads without signal degradation.

The voltage follower is fundamental in analog circuit design, ensuring signal fidelity and stability across a wide range of electronic applications. The **Flipped Voltage Follower (FVF)** is an advanced analog circuit topology derived from the conventional source follower, optimized for low-voltage, low-power applications. Unlike the standard voltage follower, the FVF employs a feedback structure that forces the input transistor to operate at a constant drain current, independent of variations in input voltage or load current. This is achieved using shunt negative feedback and ancillary biasing circuitry, resulting in improved linearity and significantly reduced output impedance compared to traditional designs.

**Key Features:**
- **Low Output Impedance:** The FVF provides much lower output impedance than conventional voltage followers, making it highly effective as a voltage buffer in demanding analog applications[9][11].
- **High Linearity:** Maintains a consistent voltage transfer characteristic across a wide range of operating conditions[5][9].
- **Large Output Current Capability:** Able to source or sink larger currents, supporting class-AB operation and driving heavier loads[4][7][10].
- **Low-Voltage Operation:** Well-suited for modern low-supply-voltage and low-power integrated circuit designs[4][5][7][10].
- **Applications:** Commonly found in output stages, current mirrors, voltage buffers, gain-boosting circuits, OTAs, filters, and VCOs[4][5][7][10][11].

The FVF is a versatile and robust building block in analog and mixed-signal circuit design, offering superior performance for buffering, level shifting, and driving loads in advanced CMOS technologies.

(a) Conventional Voltage follower (common Drain); (b) Flipped voltage follower (FVF).

![](_images/FVF.png)

```bibtex
Domala, N., Sasikala, G. Low power flipped voltage follower current mirror with improved input output impedances. Sādhanā 46, 142 (2021). https://doi.org/10.1007/s12046-021-01665-6
```

## **NetList generation and LVS**
let's go through the step by step procedure to generate LVS and DRC clean layout of a FVF cell.

In [ ]:
import os
import subprocess

# Run a shell, source .bashrc, then printenv
cmd = 'bash -c "source ~/.bashrc && printenv"'
result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
env_vars = {}
for line in result.stdout.splitlines():
    if '=' in line:
        key, value = line.split('=', 1)
        env_vars[key] = value

# Now, update os.environ with these
os.environ.update(env_vars)

In [ ]:
from glayout import MappedPDK, sky130 , gf180
#from gdsfactory.cell import cell
from gdsfactory import Component
from gdsfactory.components import text_freetype, rectangle

In [ ]:
from glayout import nmos, pmos
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring

In [ ]:
from glayout.util.comp_utils import evaluate_bbox, prec_center, prec_ref_center, align_comp_to_port
from glayout.util.port_utils import add_ports_perimeter,print_ports
from glayout.util.snap_to_grid import component_snap_to_grid
from glayout.spice.netlist import Netlist

In [ ]:
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route

FVF has two fets as shown in the schematic. We call M1 as input fet and M2 as feedback fet. Lets define arguments for the FETs

### 2. Basic Usage of the GLayout Framework
Each generator is a Python function that takes a `MappedPDK` object as a parameter and generates a DRC clean layout for the given PDK. The generator may also accept a set of optional layout parameters such as the width or length of a MOSFET. All parameters are normal Python function arguments.

The generator returns a `GDSFactory.Component` object that can be written to a `.gds` file and viewed using a tool such as Klayout. In this example, the `gdstk` library is used to convert the `.gds` file to an SVG image for viewing.

The pre-PEX SPICE netlist for the component can be viewed using `component.info['netlist'].generate_netlist()`.

In the following example the FET generator `glayout.primitives.fet` is imported and run with both the [Skywater 130](https://skywater-pdk.readthedocs.io/en/main/) and [GF180](https://gf180mcu-pdk.readthedocs.io/en/latest/) PDKs.

#### Demonstration of Basic Layout / Netlist Generation in SKY130 & GF180

In [ ]:
import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Used to display the results in a grid (notebook only)
left = widgets.Output()
leftSPICE = widgets.Output()
right = widgets.Output()
rightSPICE = widgets.Output()
hide = widgets.Output()

grid = widgets.GridspecLayout(1, 4)
grid[0, 0] = left
grid[0, 1] = leftSPICE
grid[0, 2] = right
grid[0, 3] = rightSPICE
display(grid)

def display_gds(gds_file, scale = 3):
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg('../../out.svg')

  # Scale the image for displaying
  fig = sg.fromfile('../../out.svg')
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save('../../out.svg')

  # Display the image
  IPython.display.display(IPython.display.SVG('../../out.svg'))

def display_component(component, scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds("../../out.gds")

  display_gds('../../out.gds', scale)

with hide:
  # Generate the sky130 component
  component_sky130 = nmos(pdk = sky130, fingers=5)
  # Generate the gf180 component
  component_gf180 = nmos(pdk = gf180, fingers=5,with_dnwell=False)

# Display the components' GDS and SPICE netlists
with left:
  print('Skywater 130nm N-MOSFET (fingers = 5)')
  display_component(component_sky130, scale=2)
with leftSPICE:
  print('Skywater 130nm SPICE Netlist')
  print(component_sky130.info['netlist'].generate_netlist())

with right:
  print('GF 180nm N-MOSFET (fingers = 5)')
  display_component(component_gf180, scale=2)
with rightSPICE:
  print('GF 180nm SPICE Netlist')
  print(component_gf180.info['netlist'].generate_netlist())

### This part is just to import the file you created from Tutorial 1

In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath("../../FVF"))

from my_FVF import flipped_voltage_follower,add_fvf_labels

### Creating A Netlist

In [ ]:
# This is a different way of creating netlist compared to what we already have in the repo.
# Instead of giving the low level components as input, you append them to the top_level.info in component function.
# Then you can access them as showed here.

def fvf_netlist(fvf_in: Component) -> Netlist:

    fet_1 = fvf_in.info["fet_1"]  
    fet_2 = fvf_in.info["fet_2"]
    
    netlist = Netlist(circuit_name='FLIPPED_VOLTAGE_FOLLOWER', nodes=['VIN', 'VBULK', 'VOUT', 'Ib'])
    
    netlist.connect_netlist(fet_1.info['netlist'], [('D', 'Ib'), ('G', 'VIN'), ('S', 'VOUT'), ('B', 'VBULK')])
    netlist.connect_netlist(fet_2.info['netlist'], [('D', 'VOUT'), ('G', 'Ib'), ('S', 'VBULK'), ('B', 'VBULK')])
    
    fvf_in.info['netlist'] = netlist
    
    return fvf_in

my_fvf = fvf_netlist(flipped_voltage_follower(gf180, width=(4,2.75), length=(2,1)))
my_fvf  = add_fvf_labels(my_fvf, gf180)
my_fvf .name = "FVF"
#my_fvf.write_gds('out_FVF.gds')
my_fvf.show()
print(my_fvf.info['netlist'].generate_netlist())

### Run LVS
 LVS(Layout Versus Schematic) is an automated process that compares the extracted netlist of the physical layout (how the transistors and wires are actually drawn on the silicon) against the original, intended netlist from the schematic (the circuit's functional design). Its primary goal is to ensure that the layout precisely matches the schematic, catching any discrepancies like shorts, opens, missing components, or incorrect connections. `Netgen` is the tool we use for LVS here.

In [ ]:
fvf = fvf_netlist(add_fvf_labels(flipped_voltage_follower(gf180,width=(4,2.75),length=(2,1)),gf180))
fvf.name = "fvf"
fvf_gds = fvf.write_gds("fvf.gds")
#display_gds(fvf_gds)
fvf.show()
netgen_lvs_result = gf180.lvs_netgen(fvf, fvf.name)

## Extraction

In [ ]:
#At first, we need to properly set PDKPATH and PDK_ROOT 

print("Making sure your environment varriables are still correctly set")
print(" I am using PDK in: ",os.environ['PDK_ROOT'],"\n PDK name: ",os.environ['PDK'],"\n The PDK files are at: ",os.environ['PDKPATH'])

In [ ]:
import tempfile
magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"
magicrc_file

In [ ]:
design_name=fvf.name
path_to_dir = Path("../../FVF").resolve() / "ext" / design_name
if not path_to_dir.exists():
    path_to_dir.mkdir(parents=True, exist_ok=False)

pex_path = path_to_dir / f"{design_name}_pex.spice"
gds_path = path_to_dir / f"{design_name}.gds"


fvf.write_gds(str(gds_path))
    
magic_script_content = f"""
drc off            
gds flatglob *\\$\\$*
gds read {gds_path}

flatten {design_name}
load {design_name}
select top cell
extract do local
extract all
ext2sim labels on
ext2sim
extresist tolerance 10
extresist
ext2spice lvs
ext2spice cthresh 0
ext2spice extresist on
ext2spice -o {str(pex_path)}
exit
"""

with tempfile.NamedTemporaryFile(mode='w', delete=False) as magic_script_file:
    magic_script_file.write(magic_script_content)
    magic_script_path = magic_script_file.name
    
magic_cmd = f"bash -c 'magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}'",
magic_subproc = subprocess.run(
    magic_cmd, 
    shell=True,
    check=True,
    capture_output=True
)

magic_subproc_code = magic_subproc.returncode
magic_subproc_out = magic_subproc.stdout.decode('utf-8')
print(magic_subproc_out)

#You might want to remove the gds file for size concerns
# or keep it for testing
# os.remove(magic_script_path)
# for file in os.listdir(path_to_dir):
#         if file.endswith(".gds"):
#              os.remove(file)

In [ ]:
import glob
extensions = [
            "els"
            "*.gds",
            "*.ext",
            "*.res.ext",
            "*.lvs.rpt",
            "*_lvs.rpt",
            "*.nodes",
            "*.sim",
            "*.pex.spice",
            "*_pex.spice"
            ]
files_to_delete = []
for ext in extensions:
    files_to_delete.extend(glob.glob(ext))
    
# Delete the files
for file_path in files_to_delete:
    try:
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    except OSError as e:
        print(f"Error deleting {file_path}: {e}")

### Post-PEX simulation
We do `.op` analysis and `.ac` anaysis on our fvf block.

#### `.op` analysis - find out the DC voltage at output node

In [ ]:
designs_file=str(Path(os.environ['PDKPATH']) / "libs.tech" / "ngspice" / "design.ngspice")
models_file = str(Path(os.environ['PDKPATH']) / "libs.tech" / "ngspice" / "sm141064.ngspice")

pex_file = str(pex_path).split("/")[-1]

fvf_op_tb_string=f"""* FVF Output Impedance Testbench
.temp 25
.param vcm = 1.3
.param ib = 10u

************* Power Supplies *************
Vsupply VDD GND 1.8
V0 vb GND 0

************* Input Bias (DC bias at IN) *************
VIN vin GND {{vcm}}  ; Mid-supply bias (renamed from 'IN' to 'VIN' to avoid conflict if 'IN' is a global node or subckt port)
Ibias VDD ib {{ib}}

************* DUT: FVF Subcircuit *************
**Import GF180 lib
.include {designs_file}
.lib {models_file} typical

** Import fvf subcircuit
.include {pex_file}
* Adjust node order if needed based on your fvf_pex.spice subcircuit definition.
* Assuming fvf subcircuit takes (vb, output_node, input_node, ibias_node)
XDUT vb vin FVF_OUT ib fvf

************* Analysis *************
* Operating point (for sanity check)
.op ; Uncomment this to run DC operating point analysis and check node voltages

.GLOBAL VDD 
.GLOBAL GND

.end
"""

tb1_path = path_to_dir / f"{design_name}_op_tb.sp"
tb1_path.write_text(fvf_op_tb_string, encoding="utf-8")

In [ ]:
tb1_file = "../../FVF/ext/fvf/"+str(tb1_path).split("/")[-1]

try:
    ngspice_subproc = subprocess.run(
        ['ngspice', '-b', str(tb1_file)],
        check=True,
        capture_output=True
    )
    print(ngspice_subproc.stdout.decode('utf-8'))
except subprocess.CalledProcessError as e:
    print("STDOUT:\n", e.stdout.decode())
    print("STDERR:\n", e.stderr.decode())

If you did not do any changes in testbench, the voltage at output node FVF_OUT should be around 0.45V

#### `.ac` analysis - find out the output impedance

In [ ]:
designs_file=str(Path(os.environ['PDKPATH']) / "libs.tech" / "ngspice" / "design.ngspice")
models_file = str(Path(os.environ['PDKPATH']) / "libs.tech" / "ngspice" / "sm141064.ngspice")

pex_file = str(pex_path).split("/")[-1]

fvf_zo_tb_string=f"""* FVF Output Impedance Testbench
.temp 25
.param vcm = 1.3
.param ib = 10u

************* Power Supplies *************
Vsupply VDD GND 1.8
V0 vb GND 0

************* Input Bias (DC bias at IN) *************
VIN vin GND {{vcm}}  ; Mid-supply bias (renamed from 'IN' to 'VIN' to avoid conflict if 'IN' is a global node or subckt port)
Ibias VDD ib {{ib}}

************* AC Test Source for Output Impedance Measurement *************
*Insert an AC current source at the output

I_TEST GND FVF_OUT AC 1u ; AC 1 magnitude, 0 DC offset, in series with FVF output

************* DUT: FVF Subcircuit *************
**Import GF180 lib
.include {designs_file}
.lib {models_file} typical

** Import fvf subcircuit
.include {pex_file}
* Adjust node order if needed based on your fvf_pex.spice subcircuit definition.
* Assuming fvf subcircuit takes (vb, output_node, input_node, ibias_node)
XDUT vb vin FVF_OUT ib fvf

************* AC analysis **************
* AC sweep to compute output impedance magnitude and phase
.ac dec 10 10 10G
************* Control Output *************
* Calculate Output impedance: Zout = V(FVF_OUT) / 1u
.plot AC MAG(V(FVF_OUT))/1e-6 ; Plots the magnitude of the calculated output impedance

.GLOBAL VDD 
.GLOBAL GND

.end
"""
tb2_path = path_to_dir / f"{design_name}_zo_tb.sp"
tb2_path.write_text(fvf_zo_tb_string, encoding="utf-8")


In [ ]:
tb2_file = "../../FVF/ext/fvf/"+str(tb2_path).split("/")[-1]

try:
    ngspice_subproc = subprocess.run(
        ['ngspice', '-b', str(tb2_file)],
        check=True,
        capture_output=True
    )
    print(ngspice_subproc.stdout.decode('utf-8'))
except subprocess.CalledProcessError as e:
    print("STDOUT:\n", e.stdout.decode())
    print("STDERR:\n", e.stderr.decode())

In [ ]:
# # You can also run ngspice in batch mode
# !ngspice -b fvf_zo_tb.sp